# Credit Risk — Will This Loan Be Repaid?

Classification study on the Kaggle
[credit risk dataset](https://www.kaggle.com/datasets/laotse/credit-risk-dataset)
(`laotse/credit-risk-dataset`): 32,581 granted loans, twelve columns of applicant,
credit-bureau and underwriting information.

| # | Question | Section |
| --- | --- | --- |
| 1 | The dataset has no obvious label. Which variable should be the dependent variable? | [Q1](#Q1) |
| 2 | Follow ML practice and explain the relations between the variables. | [Q2](#Q2) |
| 3 | Three classifiers, cross-validated. | [Q3](#Q3) |
| 4 | Three hyperparameters each, optimised. | [Q4](#Q4) |
| 5 | A gradient-boosting classifier, optimised. | [Q5](#Q5) |
| 6 | Dataset size, whether tuning guarantees generalisation, and how to measure robustness. | [Q6](#Q6) |

The heavy lifting lives in `src/` so this notebook and `run_analysis.py` share one
implementation rather than drifting apart. Read `src/experiments.py` for the search
spaces and `src/robustness.py` for the diagnostics.

## Setup

In [ ]:
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

sys.path.insert(0, str(Path.cwd() / "src"))
warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)
%matplotlib inline

FIGURES = Path("reports/figures")
FIGURES.mkdir(parents=True, exist_ok=True)

In [ ]:
import eda
import robustness as rb
from data import TARGET, UNDERWRITING_FEATURES, load_credit_risk
from experiments import (
    RANDOM_STATE, cross_validate_baselines, gradient_boosting_spec,
    search_specs, search_trace, tune_all,
)
from preprocess import clean, feature_columns

dataset = load_credit_risk()
frame, clean_report = clean(dataset.frame)

print(f"source            : {dataset.source}")
print(f"rows x columns    : {frame.shape[0]:,} x {frame.shape[1]}")
print(f"default rate      : {frame[TARGET].mean():.3%}")
print(f"cleaning          : {clean_report}")

> **Which data is this?** `load_credit_risk()` resolves in three steps: the real CSV in
> `data/`, then a `kagglehub` download, then a calibrated stand-in. If the line above says
> `synthetic`, Kaggle was unreachable and **the numbers in this notebook come from the
> stand-in, not the real file** — it matches the real marginals and conditional default
> rates but not its sharper joint structure. Drop the real
> `data/credit_risk_dataset.csv` in and re-run to regenerate everything.

<a id="Q1"></a>
## Q1 — Which variable is the dependent variable?

The file is a bank's operational record, not a prepared ML dataset, so the label has to be
argued for rather than looked up. A usable target must be:

1. an **outcome**, not an input to the process;
2. **unknown at decision time** — otherwise there is nothing to predict;
3. **low-cardinality** enough to be a classification label.

Only one column clears all three.

In [ ]:
display(eda.profile(frame))
display(eda.target_candidates(frame))

In [ ]:
counts = frame[TARGET].value_counts().sort_index()
print(counts.to_string(), f"\n\nbase rate = {frame[TARGET].mean():.3%}")
print(f"a 'nobody defaults' classifier scores {1 - frame[TARGET].mean():.2%} accuracy")
display(Image(str(FIGURES / "01_target_balance.png"))) if (FIGURES / "01_target_balance.png").exists() else None

### Answer

**`loan_status`** — 1 = defaulted, 0 = repaid.

Everything else is an applicant attribute (`person_*`), a term of the application
(`loan_amnt`, `loan_intent`), a credit-bureau attribute (`cb_person_*`), or the lender's own
pricing decision (`loan_grade`, `loan_int_rate`). All are known *before* repayment is.

Two consequences that shape the rest of the notebook:

- **The classes are imbalanced ~78/22**, so accuracy is not a usable score — the trivial
  "everyone repays" classifier already beats 78%. ROC-AUC is the selection metric, with
  average precision, F1, balanced accuracy and the Brier score reported alongside.
- **The assignment asks about whether a loan will be *granted*, but this file contains only
  loans that were already granted.** There are no declined applicants in it. What can
  honestly be learned is *default risk on approved loans* — an input to a granting decision,
  not the decision itself. Treating one as the other is survivorship bias; see Q6.

<a id="Q2"></a>
## Q2 — How the variables relate

Three questions: how strongly does each feature relate to the target, which features
duplicate each other, and — the one that actually matters here — which features are
*consequences* of the thing being predicted rather than causes of it.

In [ ]:
assoc = eda.association_with_target(frame)
display(assoc)

In [ ]:
for col, table in eda.default_rate_by_category(frame).items():
    print(f"\n{col}")
    print(table.to_string(index=False))

In [ ]:
display(eda.numeric_correlations(frame))
display(eda.redundancy_check(frame))

In [ ]:
eda.write_figures(frame, FIGURES)
for name in ["02_default_rate_by_category.png", "03_numeric_by_outcome.png", "04_spearman_correlation.png"]:
    display(Image(str(FIGURES / name)))

### What the structure means

**1. `loan_grade` and `loan_int_rate` are the lender's verdict, not the applicant's profile.**
They are the two strongest predictors precisely *because* an underwriter already compressed
the applicant's risk into them. That makes them **post-treatment variables**: fine if the
model runs *after* grading, but leakage-like if it is meant to *replace* grading. Q3 scores
both feature sets so the difference is visible rather than assumed.

**2. `loan_int_rate` is a lookup on `loan_grade`.** Near-deterministic. Harmless for trees;
in a linear model it splits one effect across two coefficients so neither looks important.

**3. `loan_percent_income` = `loan_amnt / person_income` by construction** — and it is the
strongest *applicant-side* signal. That is the substantive finding: what predicts default is
neither income nor loan size alone but the ratio between them. Debt service capacity.

**4. `person_age` and `cb_person_cred_hist_length` are near-duplicates** — a credit file
cannot predate adulthood. One of them is redundant.

**5. Missingness is not random.** `loan_int_rate` and `person_emp_length` both have gaps.
Imputation is fitted *inside* each CV fold, never on the full dataset, so imputed values
carry no information from the validation split.

**6. Renters default far more than owners, and a prior default roughly doubles the rate** —
collateral and past behaviour dominate, which is what the credit literature would predict.

<a id="Q3"></a>
## Q3 — Three classifiers, cross-validated

### Protocol

- A stratified **20% test set is split off first** and touched exactly once, at the end.
- **5-fold stratified cross-validation** on the remaining 80%; stratification holds the
  ~22% default rate in every fold.
- Imputation, scaling and encoding live **inside** the pipeline, re-fitted per fold.
- Three learners spanning three inductive biases: linear (**logistic regression**), local
  non-parametric (**k-NN**), non-linear ensemble (**random forest**).

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    frame.drop(columns=[TARGET]), frame[TARGET],
    test_size=0.20, stratify=frame[TARGET], random_state=RANDOM_STATE,
)
full_cols = feature_columns(include_underwriting=True)
app_cols = feature_columns(include_underwriting=False)

print(f"train {len(X_train):,} ({y_train.mean():.2%} default) | test {len(X_test):,} ({y_test.mean():.2%} default)")
print(f"\nall features        ({len(full_cols)}): {full_cols}")
print(f"application-only    ({len(app_cols)}): {app_cols}")
print(f"withheld as post-treatment: {UNDERWRITING_FEATURES}")

In [ ]:
baseline = cross_validate_baselines(X_train, y_train, full_cols)
display(baseline)

In [ ]:
baseline_app = cross_validate_baselines(X_train, y_train, app_cols)
display(baseline_app)

delta = (baseline.set_index("model")["roc_auc_mean"] - baseline_app.set_index("model")["roc_auc_mean"])
print("ROC-AUC lost when the lender's grade and rate are withheld:")
print(delta.round(4).to_string())

The drop is the share of apparent performance that comes from the underwriter's verdict
rather than from anything the applicant reported. It is smaller than one might expect,
which says the grade is largely *derived from* the same applicant attributes the model
already sees — it is a summary, not new information.

<a id="Q4"></a>
## Q4 — Hyperparameter optimisation

Three hyperparameters per classifier, each spanning a real range rather than a
neighbourhood of the default. Same folds and same seed as Q3, so the comparison is
like-for-like.

In [ ]:
specs = search_specs()
for s in specs:
    print(f"{s.name}  ({s.strategy} search{', ' + s.notes if s.notes else ''})")
    for k, why in s.tuned.items():
        print(f"    {k:20s} {str(s.param_grid['clf__' + k]):38s} {why}")
    print()

In [ ]:
tuned, searches = tune_all(specs, X_train, y_train, full_cols)
display(tuned[["model", "n_candidates", "best_params", "roc_auc_mean", "roc_auc_std",
               "pr_auc_mean", "train_roc_auc_mean", "overfit_gap", "search_seconds"]])

In [ ]:
for name, search in searches.items():
    print(f"\n=== {name} — top candidates ===")
    display(search_trace(search))

In [ ]:
comparison = pd.DataFrame({
    "baseline": baseline.set_index("model")["roc_auc_mean"],
    "tuned": tuned.set_index("model")["roc_auc_mean"],
})
comparison["delta"] = (comparison["tuned"] - comparison["baseline"]).round(4)
display(comparison)

Note the `overfit_gap` column (train AUC minus cross-validated AUC). A large gap on a model
whose *cross-validated* score still improved is the signature of a learner that memorises
the training set — k-NN with `weights='distance'` reproduces its training labels exactly.
The CV score is what matters, but the gap is worth seeing: it is the same mechanism that
makes a tuned score optimistic, which is the subject of Q6.

<a id="Q5"></a>
## Q5 — Gradient boosting

`HistGradientBoostingClassifier` — scikit-learn's histogram-based booster, the same family
as LightGBM. It fits this problem: mixed numeric and categorical features, a strongly
non-linear and interacting relationship between `loan_percent_income` and `loan_grade`, and
insensitivity to the heavy right tail of `person_income` that the linear model needs
rescaled. It runs through the same pipeline as the other three so the comparison holds.

The number of boosting rounds is **not** searched — early stopping on an internal validation
split sets it per fit, which is both cheaper and less prone to overfitting the search.

In [ ]:
gb_spec = gradient_boosting_spec()
print(f"{gb_spec.name}  ({gb_spec.strategy} search over {gb_spec.n_iter} candidates; {gb_spec.notes})")
for k, why in gb_spec.tuned.items():
    print(f"    {k:22s} {str(gb_spec.param_grid['clf__' + k]):26s} {why}")

gb_tuned, gb_searches = tune_all([gb_spec], X_train, y_train, full_cols)
gb_search = gb_searches[gb_spec.name]
best_model = gb_search.best_estimator_
display(gb_tuned[["model", "n_candidates", "best_params", "roc_auc_mean", "roc_auc_std",
                  "pr_auc_mean", "overfit_gap", "search_seconds"]])

In [ ]:
display(search_trace(gb_search, top=8))

leaderboard = pd.concat([tuned, gb_tuned], ignore_index=True).sort_values("roc_auc_mean", ascending=False)
display(leaderboard[["model", "roc_auc_mean", "roc_auc_std", "pr_auc_mean", "f1_mean", "brier_mean"]])

rb.plot_model_comparison(baseline, leaderboard, FIGURES / "05_model_comparison.png")
display(Image(str(FIGURES / "05_model_comparison.png")))

### Held-out test set

Scored once, on data no fold and no search ever saw. Intervals are 2,000-sample percentile
bootstraps.

In [ ]:
y_proba = best_model.predict_proba(X_test[full_cols])[:, 1]

boot = rb.bootstrap_test_metrics(y_test, y_proba, n_boot=2000)
display(boot)

rb.plot_evaluation(y_test, y_proba, FIGURES / "06_holdout_evaluation.png")
display(Image(str(FIGURES / "06_holdout_evaluation.png")))

In [ ]:
display(rb.threshold_table(y_test, y_proba))

Ranking quality is threshold-free; an approve/decline policy is not. This table is where the
business trade-off actually gets made — how many good borrowers you are willing to turn away
per default you avoid. No amount of hyperparameter search makes that choice.

<a id="Q6"></a>
## Q6 — Dataset size, guarantees, and robustness

### Is 32k rows a lot or a little?

A comfortable middle — but the *minority class* is the binding constraint. At a ~22% base
rate the effective sample for learning what default looks like is the ~7k defaulters, not
the 32k rows. Hence stratification everywhere, cross-validation rather than a single split,
and real caution about grades F and G, which hold only a few hundred rows between them.

The learning curve answers whether more rows would help.

In [ ]:
curve = rb.learning_curve_data(best_model, X_train, y_train, full_cols)
display(curve)

rb.plot_learning_curve(curve, FIGURES / "07_learning_curve.png")
display(Image(str(FIGURES / "07_learning_curve.png")))

slope = curve["cv_mean"].iloc[-1] - curve["cv_mean"].iloc[-2]
print(f"change in CV AUC over the last step: {slope:+.4f}")

Flat. The limit is the **information in these twelve columns**, not the number of rows.
Another 30k identical applications would buy almost nothing; new *columns* — payment
history, existing debt, a bureau DTI — would buy a lot.

### Does hyperparameter optimisation guarantee performance on unseen data?

**No — and it is systematically optimistic.** Three separate reasons.

**1. The reported best score is a maximum over noisy estimates.** A search over *k*
candidates returns the best of *k* noisy fold-averages; part of that "best" is a lucky draw
against those particular folds. Nested cross-validation measures it — an outer loop scoring
the *whole procedure*, tuning included, on data the search never touched.

In [ ]:
nested = rb.nested_cv(gb_spec, X_train, y_train, full_cols, n_iter=10)
print(f"tuned inner-CV score (what a naive report quotes) : {gb_search.best_score_:.4f}")
print(f"nested CV (honest estimate of the procedure)      : {nested['mean']:.4f} +/- {nested['std']:.4f}")
print(f"selection optimism                                : {gb_search.best_score_ - nested['mean']:+.4f}")
print(f"outer-fold scores                                 : {nested['outer_scores']}")

**2. It optimises the metric you chose, on the distribution you happen to have.** Maximising
ROC-AUC improves *ranking*. It does not improve calibration, and it knows nothing about the
cost asymmetry between rejecting a good borrower and approving a bad one.

**3. It assumes the future resembles the past.** Every guarantee cross-validation offers is
conditional on new applicants coming from the same distribution. Credit portfolios break
that routinely — rates move, the lender's own approval policy shifts, a recession arrives.
And because this file holds only *approved* loans, a model deployed to make approval
decisions immediately faces a population it never saw.

### How to measure robustness

**(a) Nested CV** — above; isolates selection bias.

**(b) Bootstrap intervals** — how precise the estimate is at all. Compare the CI width to
the gaps between models before claiming one is better.

In [ ]:
auc_ci = boot.set_index("metric").loc["roc_auc"]
spread = leaderboard["roc_auc_mean"].max() - leaderboard["roc_auc_mean"].min()
print(f"held-out ROC-AUC : {auc_ci['point_estimate']:.4f}  95% CI [{auc_ci['ci_lo_2.5']:.4f}, {auc_ci['ci_hi_97.5']:.4f}]")
print(f"CI width                        : {auc_ci['ci_width']:.4f}")
print(f"spread across all tuned models  : {spread:.4f}")
print(f"\n-> models separated by less than {auc_ci['ci_width']:.4f} AUC are not distinguishable on this data.")

**(c) Seed / partition sensitivity** — re-run CV under different fold partitions. If the
score swings with the partition, the model ranking is noise.

In [ ]:
seeds = rb.seed_sensitivity(best_model, X_train, y_train, full_cols)
display(seeds)
print(f"spread across seeds: {seeds.attrs['spread']:.4f}")

**(d) Learning curve** — above; separates "needs more rows" from "needs better features".

**(e) Slice-level evaluation** — the average hides the segments. A model can look strong
overall while being near-useless inside the segment where the decision is actually hard.

In [ ]:
subgroups = rb.subgroup_performance(X_test.reset_index(drop=True), y_test.to_numpy(), y_proba)
display(subgroups)

within = subgroups["roc_auc"].dropna()
print(f"per-slice AUC ranges {within.min():.3f} to {within.max():.3f} "
      f"(overall {auc_ci['point_estimate']:.3f})")

Within a single `loan_grade` the model has far less to work with than the headline suggests
— because the grade itself carries most of the signal. That is the practical meaning of the
post-treatment problem from Q2.

Two further checks belong in production but need data this file does not carry:

- **Out-of-time validation** — train on older vintages, test on newer. The only honest test
  of drift, and the one this dataset cannot support: it has no origination dates.
- **Stress / adversarial evaluation** — perturb inputs, or re-weight the test set toward a
  downturn population, and watch the metric move.

### What would make this deployable

- Score *rejected* applications too (reject inference), or state plainly that the model
  describes approved-loan risk only.
- Monitor **calibration** drift, not just AUC — a ranking can stay good while absolute
  probabilities drift, and the book still gets mispriced.
- Re-validate out-of-time on every new vintage; alarm on input drift rather than waiting for
  the default rate to move.

### Summary

In [ ]:
summary = pd.DataFrame([
    ("rows after cleaning", f"{len(frame):,}"),
    ("default base rate", f"{frame[TARGET].mean():.2%}"),
    ("best model", leaderboard.iloc[0]["model"]),
    ("tuned CV ROC-AUC", f"{gb_search.best_score_:.4f}"),
    ("nested CV ROC-AUC", f"{nested['mean']:.4f} +/- {nested['std']:.4f}"),
    ("selection optimism", f"{gb_search.best_score_ - nested['mean']:+.4f}"),
    ("held-out ROC-AUC", f"{auc_ci['point_estimate']:.4f} [{auc_ci['ci_lo_2.5']:.4f}, {auc_ci['ci_hi_97.5']:.4f}]"),
    ("bootstrap CI width", f"{auc_ci['ci_width']:.4f}"),
    ("seed spread", f"{seeds.attrs['spread']:.4f}"),
    ("learning-curve last step", f"{slope:+.4f}"),
    ("data source", dataset.source),
], columns=["quantity", "value"])
display(summary)